# MeshSOS — training the semantic encoder

Run this with **Runtime → Run all**. It takes about two minutes.

The job: turn a free-text emergency report into a **3-byte envelope** so a
complete authenticated SOS fits one 25-byte BLE advertising frame.

This notebook trains an encoder for that and compares it, honestly, against
a keyword baseline that needs no training at all.

**What you will see:** real training with epochs and a falling loss curve,
accuracy per field, and a comparison where **the baseline wins**. That last
part is the actual finding, not a failure of the notebook.


## 1. Setup

No GPU is needed and none is requested. This model is small enough that a
GPU would sit idle — stated plainly rather than switched on for appearances.


In [ ]:
!git clone -q https://github.com/preeti2428/MeshSOS-NLP.git 2>/dev/null || echo 'already cloned'
%cd -q MeshSOS-NLP

import sys, platform, sklearn, numpy
sys.path.insert(0, '.')
print('python      ', platform.python_version())
print('scikit-learn', sklearn.__version__)
print('numpy       ', numpy.__version__)


## 2. The data

Two generated sets, and they are deliberately **not** the same:

| Set | Used for | Why |
|---|---|---|
| `nlp/synth.py` | training | 6,000 messages, labels true by construction |
| `nlp/testset.py` | testing | 10,000 messages, different phrasing, split into five strata |

Training and testing on the same generator would report a number that means
nothing. The test set uses different wording on purpose.

**Both are generated, not real.** The real corpora (HumAID, CrisisBench) need
the envelope fields annotated by people, which is not done. Every number below
is a coverage diagnostic, not a benchmark score.


In [ ]:
from nlp.synth import generate as gen_train
from nlp.testset import generate as gen_test, STRATA

train = gen_train(6000, seed=11)
test  = gen_test(2000, seed=7)   # 2000 per stratum = 10,000 total

print(f'train: {len(train):,} messages')
print(f'test : {len(test):,} messages across {len(STRATA)} strata\n')

print('--- a training sample ---')
t, e = train[0]
print(' ', t)
print(' ->', e.to_dict(), '\n')

print('--- one test sample per stratum ---')
for st in STRATA:
    s = next(x for x in test if x.stratum == st)
    print(f'  [{st:<14}] {s.text[:70]}')


## 3. Baseline: keyword matching, zero training

Before training anything, measure what a plain lexicon achieves. If the
trained model cannot beat this, the trained model is not worth shipping.


In [ ]:
import time
from nlp.rules import RuleEncoder

rules = RuleEncoder()
t0 = time.perf_counter()
rule_pred = [rules.encode(s.text) for s in test]
rule_ms = (time.perf_counter() - t0) / len(test) * 1000

def acc(pred_envs, field, stratum=None):
    idx = [i for i, s in enumerate(test) if stratum is None or s.stratum == stratum]
    if not idx: return 0.0
    return 100 * sum(getattr(pred_envs[i], field) == getattr(test[i].gold, field)
                     for i in idx) / len(idx)

rule_envs = [r.envelope for r in rule_pred]
print(f'{"stratum":<16} {"category":>9} {"count":>8} {"mobility":>9} {"fallback":>9}')
print('-' * 56)
for st in STRATA:
    idx = [i for i, s in enumerate(test) if s.stratum == st]
    fb = 100 * sum(1 for i in idx if not rule_pred[i].should_compress) / len(idx)
    if st == 'unintelligible':
        print(f'{st:<16} {"—":>9} {"—":>8} {"—":>9} {fb:>8.1f}%')
    else:
        print(f'{st:<16} {acc(rule_envs,"category",st):>8.1f}% {acc(rule_envs,"casualty_count",st):>7.1f}% '
              f'{acc(rule_envs,"mobility",st):>8.1f}% {fb:>8.1f}%')
print('-' * 56)
print(f'{rule_ms:.3f} ms per message, 0 KB model, no training')


Read the `synonyms` row. That is the whole problem in one line: wording the
lexicon was never given. Category accuracy collapses — **and the encoder
refuses to compress 97% of those**, sending the original text instead of a
guess. It fails slow, not wrong.

That is what the trained model has to fix.


## 4. Training a neural encoder

A multi-layer perceptron over character n-gram features. Character n-grams,
not words, so romanised-Hindi spelling variants no word list anticipates
still land near each other.

Below: real epochs, and the loss after each one.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier

X_train = [t for t, _ in train]
y_train = [int(e.category) for _, e in train]

vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4),
                      max_features=3000, sublinear_tf=True)
Xv = vec.fit_transform(X_train)
print(f'feature matrix: {Xv.shape[0]:,} messages x {Xv.shape[1]:,} char n-grams\n')

clf = MLPClassifier(hidden_layer_sizes=(128,), max_iter=30,
                    random_state=0, verbose=True)
t0 = time.perf_counter()
clf.fit(Xv, y_train)
print(f'\ntrained in {time.perf_counter() - t0:.1f}s over {len(clf.loss_curve_)} epochs')


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 3.6), dpi=130)
ax.plot(range(1, len(clf.loss_curve_) + 1), clf.loss_curve_,
        color='#2563EB', linewidth=2)
ax.set_xlabel('epoch'); ax.set_ylabel('training loss')
ax.set_title('Training loss', loc='left', fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', color='#E3E1DA', linewidth=0.8)
ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

print(f'loss: {clf.loss_curve_[0]:.4f} -> {clf.loss_curve_[-1]:.4f}')


The loss falls almost to zero. On its own that looks like success.

It is not. It means the model has learned the training generator very well.
The next cell tests it on text that generator never produced.


## 5. The comparison that matters


In [ ]:
import numpy as np

X_test = [s.text for s in test]
y_test = np.array([int(s.gold.category) for s in test])
pred   = clf.predict(vec.transform(X_test))

print(f'{"stratum":<16} {"neural":>9} {"rules":>9}   winner')
print('-' * 50)
rows = []
for st in STRATA:
    if st == 'unintelligible':
        continue
    idx = [i for i, s in enumerate(test) if s.stratum == st]
    nn = 100 * float((pred[idx] == y_test[idx]).mean())
    rb = acc(rule_envs, 'category', st)
    rows.append((st, nn, rb))
    print(f'{st:<16} {nn:>8.1f}% {rb:>8.1f}%   {"neural" if nn > rb else "rules"}')

idx_all = [i for i, s in enumerate(test) if s.stratum != 'unintelligible']
nn_all = 100 * float((pred[idx_all] == y_test[idx_all]).mean())
# Exclude the unintelligible stratum from both: gold is OTHER there and
# the encoder answers OTHER, which would inflate the figure.
rb_all = 100 * sum(int(rule_envs[i].category) == y_test[i] for i in idx_all) / len(idx_all)
print('-' * 50)
print(f'{"OVERALL":<16} {nn_all:>8.1f}% {rb_all:>8.1f}%')


In [ ]:
labels = [r[0] for r in rows]
nn_v   = [r[1] for r in rows]
rb_v   = [r[2] for r in rows]
x = np.arange(len(labels)); w = 0.36

fig, ax = plt.subplots(figsize=(7.5, 3.8), dpi=130)
b1 = ax.bar(x - w/2, nn_v, w, label='neural (trained)', color='#2563EB', edgecolor='white')
b2 = ax.bar(x + w/2, rb_v, w, label='rules (no training)', color='#B45309', edgecolor='white')
for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 1.5,
                f'{b.get_height():.0f}', ha='center', fontsize=8.5)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('category accuracy (%)'); ax.set_ylim(0, 100)
ax.set_title('Trained model vs keyword baseline', loc='left', fontweight='bold')
ax.legend(frameon=False, fontsize=9)
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.grid(axis='y', color='#E3E1DA', linewidth=0.8); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()


## 6. What the comparison actually says

Not "the model lost". Look at it per stratum:

- Where the lexicon has the words, **rules win** — they encode the right
  answer directly instead of inferring it.
- On **synonyms**, where the lexicon has nothing, the model wins by roughly
  **five times**. Character n-grams reach wording no keyword list contains.

They fail in opposite places. So use both.


## 7. Cascade: rules first, model for the rest

The rule encoder already says when it is unsure — that is the confidence
gate that decides whether to compress at all. So route on it:

```
text ─→ rules ──confident──→ 3-byte envelope
          │
          └──not confident──→ model ──→ 3-byte envelope
```

Nothing new has to be trained. The routing signal was already there.


In [ ]:
cascade = np.array([
    int(rule_envs[i].category) if rule_pred[i].should_compress else int(pred[i])
    for i in range(len(test))
])
rules_cat = np.array([int(e.category) for e in rule_envs])

print(f'{"stratum":<16} {"rules":>8} {"neural":>8} {"CASCADE":>9}')
print('-' * 45)
casc_rows = []
for st in STRATA:
    if st == 'unintelligible':
        continue
    i = [k for k, s in enumerate(test) if s.stratum == st]
    r_, n_, c_ = (100 * (rules_cat[i] == y_test[i]).mean(),
                  100 * (pred[i] == y_test[i]).mean(),
                  100 * (cascade[i] == y_test[i]).mean())
    casc_rows.append((st, r_, n_, c_))
    print(f'{st:<16} {r_:>7.1f}% {n_:>7.1f}% {c_:>8.1f}%')

i = idx_all
print('-' * 45)
print(f'{"OVERALL":<16} {100*(rules_cat[i]==y_test[i]).mean():>7.1f}% '
      f'{100*(pred[i]==y_test[i]).mean():>7.1f}% '
      f'{100*(cascade[i]==y_test[i]).mean():>8.1f}%')

g = [k for k, s in enumerate(test) if s.stratum == 'unintelligible']
refuse = 100 * sum(1 for k in g if not rule_pred[k].should_compress) / len(g)
print(f'\nunintelligible input still refused: {refuse:.1f}%')


In [ ]:
labels = [r[0] for r in casc_rows]
x = np.arange(len(labels)); w = 0.26
fig, ax = plt.subplots(figsize=(8, 3.9), dpi=130)
for off, key, lab, col in ((-w, 1, 'rules', '#B45309'),
                           (0.0, 2, 'neural', '#2563EB'),
                           (w,   3, 'cascade', '#059669')):
    vals = [r[key] for r in casc_rows]
    bars = ax.bar(x + off, vals, w, label=lab, color=col, edgecolor='white')
    for b in bars:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 1.5,
                f'{b.get_height():.0f}', ha='center', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('category accuracy (%)'); ax.set_ylim(0, 100)
ax.set_title('Rules and the model fail in opposite places', loc='left', fontweight='bold')
ax.legend(frameon=False, fontsize=9, ncol=3)
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.grid(axis='y', color='#E3E1DA', linewidth=0.8); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()


## 8. What this shows

**The trained model reached near-zero training loss and still lost overall.**
It learned the generator, not the task. A system evaluated only on its own
held-out split would have reported the high number and shipped the weaker
encoder. That is the first finding, and it is a warning about method, not
about neural networks.

**But the two approaches fail in opposite places,** and routing between them
on the confidence the rule encoder already produces beats either alone — with
no extra training and no new signal. The out-of-vocabulary column, which was
the whole weakness, improves several times over.

**Refusal survives the cascade.** Unintelligible input is still refused 100%
of the time, so the safety property does not depend on which encoder answered.

### Honest limits

Both training and test text are generated. The labels are exact because a
generator chose them, and real emergency reports are messier than any
generator. These numbers diagnose coverage; they are not a benchmark result.

The next step is not a bigger model. It is **HumAID / CrisisBench with the
envelope fields annotated by people**, and pretrained subword embeddings in
place of this model — those need no labelled data and target the synonyms
column directly.
